# LeetCode 591 — Tag Validator bằng Python

- Link: https://leetcode.com/problems/tag-validator/
- Độ khó: **Hard**
- Chủ đề: Xử lý chuỗi, Stack, Phân tích cú pháp (parsing), Đệ quy, Biểu thức chính quy (regex)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def isValid(self, code: str) -> bool:

```

## 1. Giới thiệu & mục tiêu học

Bài này không cần thuật toán cao siêu — cái khó là **đọc đặc tả dài và chuyển chính xác thành code**. Đây là kỹ năng cực kỳ thực tế: viết bộ kiểm tra cú pháp (HTML/XML, JSON, ngôn ngữ cấu hình...). Mấu chốt là **thứ tự kiểm tra** khi gặp ký tự `<`.

Sau notebook này bạn sẽ:

1. Biết cách **tách một đặc tả dài thành các quy tắc nhỏ** có thể kiểm tra từng cái.
2. Dùng **stack** để kiểm tra thẻ mở/đóng lồng nhau đúng (giống bài ngoặc đúng, nhưng "ngoặc" là tên thẻ).
3. Viết 3 lời giải: **rút gọn lặp bằng regex**, **quét tuyến tính với stack**, và **phân tích cú pháp đệ quy (recursive descent)**.
4. Xử lý cẩn thận CDATA, thẻ không hợp lệ, `<` lẻ loi, nội dung nằm ngoài thẻ gốc.
5. Kiểm chứng bằng cách cho 3 cài đặt độc lập **đối chiếu nhau** trên hàng chục nghìn chuỗi ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho chuỗi `code` là một đoạn mã. Viết bộ kiểm tra trả về `True` nếu mã hợp lệ theo các quy tắc sau:

1. Toàn bộ mã phải được **bọc trong một thẻ đóng hợp lệ** (valid closed tag). Ngoài thẻ đó không được có gì khác.
2. **Thẻ đóng (closed tag)** có dạng `<TAG_NAME>TAG_CONTENT</TAG_NAME>`: thẻ mở và thẻ đóng phải **cùng tên**. Thẻ đóng hợp lệ ⇔ `TAG_NAME` và `TAG_CONTENT` đều hợp lệ.
3. **`TAG_NAME` hợp lệ** ⇔ chỉ gồm **chữ in hoa**, độ dài từ **1 đến 9**.
4. **`TAG_CONTENT` hợp lệ** ⇔ có thể chứa các thẻ đóng hợp lệ khác, **CDATA**, và ký tự bất kỳ, **NGOẠI TRỪ**: `<` không khớp, thẻ mở/đóng không khớp, thẻ đóng có tên không hợp lệ.
5. Thẻ mở không khớp nếu không có thẻ đóng cùng tên (và ngược lại); các thẻ phải **lồng nhau đúng**.
6. Một `<` là **không khớp** nếu không tìm được `>` phía sau. Khi gặp `<` hoặc `</`, **mọi ký tự sau đó tới `>` gần nhất** được coi là `TAG_NAME` (dù có hợp lệ hay không).
7. **CDATA** có dạng `<![CDATA[CDATA_CONTENT]]>`. `CDATA_CONTENT` là mọi ký tự giữa `<![CDATA[` và **`]]>` đầu tiên** sau nó — nội dung này **không được phân tích** (chứa `<DIV>` cũng chỉ là chữ).

**Ví dụ (đúng)**

```text
Input:  code = "<DIV>This is the first line <![CDATA[<div>]]></DIV>"
Output: true
Giải thích: bọc trong <DIV>...</DIV>; "<div>" nằm trong CDATA nên chỉ là chữ.

Input:  code = "<DIV>>>  ![cdata[]] <![CDATA[<div>]>]]>]]>>]</DIV>"
Output: true
Giải thích: ">>  ![cdata[]] " là chữ thường; CDATA_CONTENT là "<div>]>" (tới "]]>" đầu tiên);
            phần "]]>>]" sau đó là chữ thường.
```

**Ví dụ (sai)**

```text
"<A>  <B> </A>   </B>"                       -> false  (lồng sai: B mở sau A nhưng đóng sau A)
"<DIV>  div tag is not closed  <DIV>"        -> false  (thẻ DIV không có thẻ đóng)
"<DIV>  unmatched <  </DIV>"                 -> false  (dấu < lẻ: "<  </DIV" bị coi là tên thẻ)
"<DIV> closed tags with invalid tag name  <b>123</b> </DIV>"  -> false  (tên "b" không in hoa)
"<DIV>  unmatched start tag <B>  and unmatched end tag </C>  </DIV>"  -> false
```

**Ràng buộc**

- `1 <= code.length <= 500`
- `code` gồm chữ cái tiếng Anh, chữ số, `'<'`, `'>'`, `'/'`, `'!'`, `'['`, `']'`, `'.'` và dấu cách `' '`.

In [1]:
EXAMPLES = [
    ("<DIV>This is the first line <![CDATA[<div>]]></DIV>", True),
    ("<DIV>>>  ![cdata[]] <![CDATA[<div>]>]]>]]>>]</DIV>", True),
    ("<A>  <B> </A>   </B>", False),
    ("<DIV>  div tag is not closed  <DIV>", False),
    ("<DIV>  unmatched <  </DIV>", False),
    ("<DIV> closed tags with invalid tag name  <b>123</b> </DIV>", False),
    ("<DIV> unmatched tags with invalid tag name  </1234567890> and <CDATA[[]]>  </DIV>", False),
    ("<DIV>  unmatched start tag <B>  and unmatched end tag </C>  </DIV>", False),
]
for code, exp in EXAMPLES:
    print(f"{str(exp):>5} | {code}")

 True | <DIV>This is the first line <![CDATA[<div>]]></DIV>
 True | <DIV>>>  ![cdata[]] <![CDATA[<div>]>]]>]]>>]</DIV>
False | <A>  <B> </A>   </B>
False | <DIV>  div tag is not closed  <DIV>
False | <DIV>  unmatched <  </DIV>
False | <DIV> closed tags with invalid tag name  <b>123</b> </DIV>
False | <DIV> unmatched tags with invalid tag name  </1234567890> and <CDATA[[]]>  </DIV>
False | <DIV>  unmatched start tag <B>  and unmatched end tag </C>  </DIV>


## 3. Kiến thức nền

### 3.1. `str.startswith(prefix, pos)` và `str.find(sub, start)`

- `code.startswith("<![CDATA[", i)`: kiểm tra tại vị trí `i` có chuỗi đó không — **không cần cắt chuỗi**.
- `code.find(">", j)`: vị trí đầu tiên của `">"` tính từ `j`, hoặc **`-1`** nếu không có. Quên kiểm tra `-1` là lỗi kinh điển.

In [2]:
code = "<A><![CDATA[x]]></A>"
print(code.startswith("<![CDATA[", 3), code.startswith("<![CDATA[", 0))
print(code.find("]]>", 3), code.find("zzz"))
print(code[code.find("<", 1):code.find(">", 1) + 1])

True False
13 -1



### 3.2. Kiểm tra tên thẻ: `isupper`, `isalpha` và cạm bẫy

`"ABC".isupper()` là `True`, nhưng `"A1".isupper()` cũng `True` (chữ số không có hoa/thường)! An toàn nhất là kiểm tra **từng ký tự** nằm trong `'A'..'Z'`.

In [3]:
def valid_name(s):
    return 1 <= len(s) <= 9 and all('A' <= c <= 'Z' for c in s)

for s in ["DIV", "A1", "div", "", "ABCDEFGHI", "ABCDEFGHIJ", "A B"]:
    print(f"{s!r:>13}: isupper={s.isupper()!s:>5} | valid_name={valid_name(s)}")

        'DIV': isupper= True | valid_name=True
         'A1': isupper= True | valid_name=False
        'div': isupper=False | valid_name=False
           '': isupper=False | valid_name=False
  'ABCDEFGHI': isupper= True | valid_name=True
 'ABCDEFGHIJ': isupper= True | valid_name=False
        'A B': isupper= True | valid_name=False


### 3.3. Stack cho cặp mở/đóng

Kiểm tra ngoặc đúng: gặp mở → `push`; gặp đóng → đỉnh stack phải là mở tương ứng, rồi `pop`. Cuối cùng stack phải rỗng. Với thẻ, "ngoặc" là **tên thẻ**.

In [4]:
def brackets_ok(s):
    pair = {')': '(', ']': '['}
    st = []
    for ch in s:
        if ch in '([':
            st.append(ch)
        elif ch in pair:
            if not st or st[-1] != pair[ch]:
                return False
            st.pop()
    return not st

print(brackets_ok("([])"), brackets_ok("([)]"), brackets_ok("(("))

True False False


### 3.4. Regex: lười (`.*?`), nhóm bắt và tham chiếu ngược `\1`

- `.*?` khớp **ít nhất có thể** → `<!\[CDATA\[.*?\]\]>` dừng ở `]]>` **đầu tiên** (đúng quy tắc 7).
- `<([A-Z]{1,9})>` bắt tên thẻ vào nhóm 1; `</\1>` yêu cầu thẻ đóng **cùng tên** đúng như đã bắt.
- `[^<]*` = dãy ký tự **không chứa `<`** → chỉ khớp thẻ **trong cùng** (không còn thẻ con).

In [5]:
import re
s = "a<![CDATA[x]]>b]]>c"
print(re.findall(r"<!\[CDATA\[.*?\]\]>", s), "| tham lam:", re.findall(r"<!\[CDATA\[.*\]\]>", s))
print(re.sub(r"<([A-Z]{1,9})>[^<]*</\1>", "T", "<A><B>hi</B><C></C></A>"))
print(re.fullmatch(r"<([A-Z]{1,9})>[^<]*</\1>", "<AB>x</AC>"))

['<![CDATA[x]]>'] | tham lam: ['<![CDATA[x]]>b]]>']
<A>TT</A>
None


### 3.5. Đệ quy trả về vị trí (parser kiểu "recursive descent")

Một hàm phân tích `parse_X(i)` đọc một cấu trúc `X` bắt đầu tại `i`, trả về **vị trí ngay sau** cấu trúc đó (hoặc `-1` nếu lỗi). Cấu trúc lồng nhau → hàm gọi lại chính nó.

In [6]:
def parse_parens(s, i):
    """Đọc một cụm '( ... )' lồng nhau tại i; trả vị trí sau ')' hoặc -1."""
    if i >= len(s) or s[i] != '(':
        return -1
    i += 1
    while i < len(s) and s[i] == '(':
        i = parse_parens(s, i)
        if i < 0:
            return -1
    return i + 1 if i < len(s) and s[i] == ')' else -1

print(parse_parens("(()())", 0), parse_parens("(()", 0))

6 -1


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Khi gặp `<`, có đúng 3 khả năng — và **thứ tự kiểm tra quan trọng**

| Nếu tại `i` bắt đầu bằng… | Thì đó là… | Xử lý |
|---|---|---|
| `<![CDATA[` | CDATA | nhảy tới sau `]]>` đầu tiên (không có → sai) |
| `</` | thẻ đóng | tên = tới `>` gần nhất; phải khớp đỉnh stack |
| `<` (còn lại) | thẻ mở | tên = tới `>` gần nhất; phải hợp lệ; push |
| ký tự khác | chữ thường | bỏ qua (nhưng phải đang ở trong một thẻ) |

Phải kiểm tra **CDATA trước**, vì `<![CDATA[` cũng bắt đầu bằng `<`; nếu coi là thẻ mở thì tên `"![CDATA[..."` sẽ không hợp lệ.

### 4.2. Tính tay `"<A><![CDATA[</A>]]></A>"`

| i | thấy | hành động | stack |
|---|---|---|---|
| 0 | `<A>` | thẻ mở `A` hợp lệ | `[A]` |
| 3 | `<![CDATA[` | tìm `]]>` từ vị trí 12 → ở 16; nhảy tới 19 | `[A]` |
| 19 | `</A>` | thẻ đóng `A` khớp đỉnh | `[]` |
| 23 | hết chuỗi | stack rỗng | → **true** |

`</A>` bên trong CDATA **không** được hiểu là thẻ đóng.

### 4.3. Quy tắc "bọc trong một thẻ" (quy tắc 1)

Stack **không bao giờ được rỗng** khi vẫn còn ký tự chưa đọc, trừ lúc bắt đầu (`i = 0`, phải là thẻ mở). Ví dụ `"<A></A><B></B>"`: sau `</A>` stack rỗng nhưng còn `<B>` → sai.

In [7]:
code = "<A><![CDATA[</A>]]></A>"
for i in [0, 3, 12, 16, 19]:
    print(i, repr(code[i:i + 9]))
print("vị trí ']]>' tìm từ 12:", code.find("]]>", 12))

0 '<A><![CDA'
3 '<![CDATA['
12 '</A>]]></'
16 ']]></A>'
19 '</A>'
vị trí ']]>' tìm từ 12: 16


## 5. Cách 1 — Rút gọn lặp bằng regex (baseline)

**Ý tưởng.** Giống trò "xếp gạch": liên tục **xóa thẻ trong cùng** cho tới khi không xóa được nữa.

1. Thay mọi CDATA (tới `]]>` đầu tiên) bằng một ký tự vô hại (`-`). Đồng thời thay mọi chữ `t` thường bằng `-` để ký hiệu `t` ở bước sau không bị nhầm với chữ trong mã.
2. Lặp: thay mọi `<NAME>nội dung không có <</NAME>` (thẻ trong cùng hợp lệ) bằng `t`.
3. Mã hợp lệ ⇔ cuối cùng chỉ còn đúng chuỗi `"t"` (một thẻ gốc duy nhất).

Mọi `<` không thuộc thẻ hợp lệ hoặc CDATA sẽ không bao giờ bị xóa → kết quả khác `"t"`.

In [8]:
import re

def is_valid_regex(code: str) -> bool:
    code = re.sub(r"<!\[CDATA\[.*?\]\]>|t", "-", code)
    prev = None
    while code != prev:
        prev = code
        code = re.sub(r"<([A-Z]{1,9})>[^<]*</\1>", "t", code)
    return code == "t"

for code, exp in EXAMPLES:
    assert is_valid_regex(code) == exp, code
print("Cách 1 đúng với mọi ví dụ")

Cách 1 đúng với mọi ví dụ


**Vì sao xử lý CDATA trước được?** CDATA chỉ hợp lệ bên trong thẻ; nếu nó nằm ngoài thẻ gốc thì sau khi thay bằng `-`, chuỗi cuối cùng còn `-` cạnh `t` → không phải `"t"` → sai, đúng như đề.

**Độ phức tạp:** mỗi vòng `re.sub` là `O(n)`, mỗi vòng xóa ít nhất một tầng lồng → tối đa `O(n)` vòng → **`O(n^2)`**. Với `n <= 500` chạy tốt, nhưng khó chứng minh đúng và khó gỡ lỗi.

**Nút thắt:** đọc đi đọc lại chuỗi nhiều lần. Một lần quét trái → phải với stack là đủ.

In [9]:
def trace_regex(code):
    code = re.sub(r"<!\[CDATA\[.*?\]\]>|t", "-", code)
    print("sau khi thay CDATA:", repr(code))
    prev = None
    while code != prev:
        prev = code
        code = re.sub(r"<([A-Z]{1,9})>[^<]*</\1>", "t", code)
        if code != prev:
            print("rút gọn ->         ", repr(code))
    return code == "t"

print(trace_regex("<A><B>x<![CDATA[<C>]]></B><D></D></A>"))

sau khi thay CDATA: '<A><B>x-</B><D></D></A>'
rút gọn ->          '<A>tt</A>'
rút gọn ->          't'
True


## 6. Bước đột phá

**Insight 1 — Một lần quét là đủ.** Đọc trái → phải, tại mỗi vị trí chỉ cần biết "đang mở những thẻ nào" (stack tên thẻ). Mỗi cấu trúc (thẻ mở, thẻ đóng, CDATA) được nhận diện bằng **tiền tố** tại vị trí hiện tại, và ta **nhảy qua** cả cấu trúc bằng `find`.

**Insight 2 — Quy tắc 6 làm việc phân tích rất đơn định.** Gặp `<` (không phải CDATA) thì tên thẻ luôn là đoạn tới `>` **gần nhất** — không cần đoán, không cần quay lui. Tên đó hợp lệ hay không quyết định luôn đúng/sai.

**Invariant khi quét:** tại mọi vị trí `i > 0` còn ký tự, stack **khác rỗng** (đang nằm trong thẻ gốc). Hết chuỗi thì stack phải rỗng.

**Quy tắc nếu…thì…:**
- Nếu `i > 0` và stack rỗng → sai (có gì đó ngoài thẻ gốc).
- Nếu gặp CDATA mà stack rỗng → sai (CDATA ngoài thẻ).
- Nếu không tìm thấy `>` (hoặc `]]>` cho CDATA) → sai.
- Nếu thẻ mở có tên không hợp lệ → sai.
- Nếu thẻ đóng không khớp đỉnh stack → sai.
- Nếu là chữ thường mà stack rỗng → sai.

## 7. Các cách tiếp theo

Notebook trình bày **3 cách**:

| # | Cách | Ý chính |
|---|---|---|
| 1 | Regex rút gọn lặp (mục 5) | xóa thẻ trong cùng cho tới khi còn `"t"` |
| 2 | Quét với stack | một lần trái → phải, stack tên thẻ |
| 3 | Recursive descent | hàm `parse_tag(i)` đệ quy theo đúng ngữ pháp |

### 7.1. Cách 2 — Quét tuyến tính với stack

**Giải thích.** Cài đúng bảng ở mục 4.1 cộng các quy tắc ở mục 6.

In [10]:
def is_valid_stack(code: str) -> bool:
    st = []
    i, n = 0, len(code)
    while i < n:
        if i > 0 and not st:                         # đã đóng thẻ gốc mà vẫn còn ký tự
            return False
        if code.startswith("<![CDATA[", i):          # 1) CDATA — kiểm tra TRƯỚC
            if not st:
                return False
            j = code.find("]]>", i + 9)
            if j == -1:
                return False
            i = j + 3
        elif code.startswith("</", i):               # 2) thẻ đóng
            j = code.find(">", i + 2)
            if j == -1:
                return False
            name = code[i + 2:j]
            if not st or st[-1] != name:
                return False
            st.pop()
            i = j + 1
        elif code[i] == "<":                         # 3) thẻ mở
            j = code.find(">", i + 1)
            if j == -1:
                return False
            name = code[i + 1:j]
            if not valid_name(name):
                return False
            st.append(name)
            i = j + 1
        else:                                        # 4) chữ thường
            if not st:
                return False
            i += 1
    return not st

for code, exp in EXAMPLES:
    assert is_valid_stack(code) == exp, code
print("Cách 2 đúng với mọi ví dụ")

Cách 2 đúng với mọi ví dụ


**Đọc code từng nhánh:**

- `i + 9`: độ dài `"<![CDATA["` là 9 → tìm `]]>` **sau** phần mở đầu.
- Thẻ đóng: không cần kiểm tra tên hợp lệ riêng — nếu tên khớp đỉnh stack thì nó đã hợp lệ lúc push; nếu không khớp thì đằng nào cũng sai.
- Thẻ mở: `valid_name` kiểm tra 1..9 chữ in hoa. Tên rỗng (`"<>"`) cũng bị loại.
- Nhánh chữ thường `if not st` bắt trường hợp mã bắt đầu bằng chữ (ví dụ `" <A></A>"` hoặc `"x"`); vì tại `i = 0` dòng kiểm tra đầu vòng bị bỏ qua.

**Code nộp LeetCode (Cách 2):**

In [11]:
class SolutionStack:
    def isValid(self, code: str) -> bool:
        st, i, n = [], 0, len(code)
        while i < n:
            if i > 0 and not st:
                return False
            if code.startswith("<![CDATA[", i):
                j = code.find("]]>", i + 9)
                if not st or j == -1:
                    return False
                i = j + 3
            elif code.startswith("</", i):
                j = code.find(">", i + 2)
                if j == -1 or not st or st[-1] != code[i + 2:j]:
                    return False
                st.pop()
                i = j + 1
            elif code[i] == "<":
                j = code.find(">", i + 1)
                name = code[i + 1:j]
                if j == -1 or not (1 <= len(name) <= 9 and all("A" <= c <= "Z" for c in name)):
                    return False
                st.append(name)
                i = j + 1
            else:
                if not st:
                    return False
                i += 1
        return not st

for code, exp in EXAMPLES:
    assert SolutionStack().isValid(code) == exp
print("SolutionStack OK")

SolutionStack OK


Lưu ý trong nhánh thẻ mở: khi `j == -1`, `code[i+1:-1]` vẫn chạy (không lỗi) nhưng ta trả `False` ngay vì điều kiện `j == -1` được xét trước trong `or`.

**Độ phức tạp Cách 2:** mỗi ký tự được `startswith`/`find` đọc qua một số lần hằng số (mỗi `find` bắt đầu từ vị trí hiện tại và ta nhảy tới kết quả) → **`O(n)`** thời gian (cộng chi phí so sánh tên ≤ 9 ký tự), **`O(n)`** bộ nhớ cho stack.

### 7.2. Cách 3 — Phân tích cú pháp đệ quy (recursive descent)

**Giải thích.** Viết lại đề thành **ngữ pháp**:

```text
code        := closed_tag  (và phải dùng hết chuỗi)
closed_tag  := '<' NAME '>' content '</' NAME '>'      (hai NAME giống nhau, NAME hợp lệ)
content     := ( closed_tag | cdata | ký tự khác '<' )*
cdata       := '<![CDATA[' ... ']]>'                   (tới ']]>' đầu tiên)
```

Mỗi luật thành một đoạn code: `parse_tag(i)` đọc một `closed_tag` bắt đầu ở `i`, trả về vị trí ngay sau nó hoặc `-1`. Trong phần `content`, gặp thẻ mở thì **gọi đệ quy**.

In [12]:
def is_valid_rd(code: str) -> bool:
    n = len(code)

    def parse_tag(i):
        # '<' NAME '>'
        if not code.startswith("<", i) or code.startswith("</", i):
            return -1
        j = code.find(">", i + 1)
        if j == -1:
            return -1
        name = code[i + 1:j]
        if not valid_name(name):
            return -1
        i = j + 1
        # content, cho tới khi gặp '</'
        while True:
            if i >= n:
                return -1                          # hết chuỗi mà chưa đóng
            if code.startswith("</", i):           # '</' NAME '>'
                j = code.find(">", i + 2)
                if j == -1 or code[i + 2:j] != name:
                    return -1
                return j + 1
            if code.startswith("<![CDATA[", i):
                j = code.find("]]>", i + 9)
                if j == -1:
                    return -1
                i = j + 3
            elif code[i] == "<":
                i = parse_tag(i)                    # thẻ con
                if i == -1:
                    return -1
            else:
                i += 1

    return parse_tag(0) == n

for code, exp in EXAMPLES:
    assert is_valid_rd(code) == exp, code
print("Cách 3 đúng với mọi ví dụ")

Cách 3 đúng với mọi ví dụ


**Đọc code:**

- `parse_tag(0) == n`: thẻ gốc phải bắt đầu ở 0 **và** kết thúc đúng ở cuối chuỗi (quy tắc 1).
- Trong content, `</` luôn kết thúc thẻ hiện tại: nếu tên khác `name` → sai (đây chính là "thẻ đóng không khớp" hoặc "lồng sai").
- Gặp `<` khác → gọi `parse_tag` cho thẻ con; nếu là `<` lẻ hoặc tên sai, `parse_tag` trả `-1`.
- Độ sâu đệ quy ≤ số thẻ lồng nhau ≤ `500 / 7` (thẻ ngắn nhất `<A></A>` dài 7) — an toàn.

**Code nộp LeetCode (Cách 3):**

In [13]:
class SolutionRD:
    def isValid(self, code: str) -> bool:
        n = len(code)

        def ok_name(s):
            return 1 <= len(s) <= 9 and all("A" <= c <= "Z" for c in s)

        def parse_tag(i: int) -> int:
            if not code.startswith("<", i) or code.startswith("</", i):
                return -1
            j = code.find(">", i + 1)
            if j == -1 or not ok_name(code[i + 1:j]):
                return -1
            name, i = code[i + 1:j], j + 1
            while i < n:
                if code.startswith("</", i):
                    j = code.find(">", i + 2)
                    return j + 1 if j != -1 and code[i + 2:j] == name else -1
                if code.startswith("<![CDATA[", i):
                    j = code.find("]]>", i + 9)
                    if j == -1:
                        return -1
                    i = j + 3
                elif code[i] == "<":
                    i = parse_tag(i)
                    if i == -1:
                        return -1
                else:
                    i += 1
            return -1

        return parse_tag(0) == n

for code, exp in EXAMPLES:
    assert SolutionRD().isValid(code) == exp
print("SolutionRD OK")

SolutionRD OK


**Độ phức tạp Cách 3:** mỗi ký tự được đọc hằng số lần → **`O(n)`** thời gian; bộ nhớ `O(độ sâu lồng)` cho stack đệ quy.

## 8. TRACE từng bước

Trace **Cách 2** với `code = "<A><B>x<![CDATA[</B>]]></B>y</A>"`:

| i | nhận diện | chi tiết | stack sau |
|---|---|---|---|
| 0 | thẻ mở | tên `A` | `[A]` |
| 3 | thẻ mở | tên `B` | `[A, B]` |
| 6 | chữ | `x` | `[A, B]` |
| 7 | CDATA | `]]>` ở 20 → nhảy tới 23 | `[A, B]` |
| 23 | thẻ đóng | `B` khớp | `[A]` |
| 27 | chữ | `y` | `[A]` |
| 28 | thẻ đóng | `A` khớp | `[]` |
| 32 | hết | stack rỗng | **true** |

In [14]:
def trace_stack(code):
    st, i, n = [], 0, len(code)
    print(f"code = {code!r} (n = {n})")
    while i < n:
        if i > 0 and not st:
            print(f"  i={i}: stack rỗng nhưng còn ký tự -> False")
            return False
        if code.startswith("<![CDATA[", i):
            j = code.find("]]>", i + 9)
            if not st or j == -1:
                print(f"  i={i}: CDATA lỗi -> False")
                return False
            print(f"  i={i:>2}: CDATA, nội dung {code[i + 9:j]!r}, nhảy tới {j + 3:<3} stack={st}")
            i = j + 3
        elif code.startswith("</", i):
            j = code.find(">", i + 2)
            name = code[i + 2:j] if j != -1 else None
            if j == -1 or not st or st[-1] != name:
                print(f"  i={i:>2}: thẻ đóng {name!r} không khớp đỉnh {st[-1] if st else None!r} -> False")
                return False
            st.pop()
            print(f"  i={i:>2}: thẻ đóng {name!r} khớp,                stack={st}")
            i = j + 1
        elif code[i] == "<":
            j = code.find(">", i + 1)
            name = code[i + 1:j] if j != -1 else None
            if j == -1 or not valid_name(name):
                print(f"  i={i:>2}: thẻ mở có tên {name!r} không hợp lệ -> False")
                return False
            st.append(name)
            print(f"  i={i:>2}: thẻ mở {name!r},                       stack={st}")
            i = j + 1
        else:
            if not st:
                print(f"  i={i}: chữ nằm ngoài thẻ -> False")
                return False
            i += 1
    print(f"  hết chuỗi, stack={st} ->", not st)
    return not st

assert trace_stack("<A><B>x<![CDATA[</B>]]></B>y</A>") is True

code = '<A><B>x<![CDATA[</B>]]></B>y</A>' (n = 32)
  i= 0: thẻ mở 'A',                       stack=['A']
  i= 3: thẻ mở 'B',                       stack=['A', 'B']
  i= 7: CDATA, nội dung '</B>', nhảy tới 23  stack=['A', 'B']
  i=23: thẻ đóng 'B' khớp,                stack=['A']
  i=28: thẻ đóng 'A' khớp,                stack=[]
  hết chuỗi, stack=[] -> True


Trace một số chuỗi **sai** để thấy mỗi quy tắc bắt lỗi ở đâu:

In [15]:
for code in ["<A>  <B> </A>   </B>", "<DIV>  unmatched <  </DIV>", "<A></A><B></B>", "<A><![CDATA[x]]</A>", "<AAAAAAAAAA></AAAAAAAAAA>"]:
    assert trace_stack(code) is False
    print()

code = '<A>  <B> </A>   </B>' (n = 20)
  i= 0: thẻ mở 'A',                       stack=['A']
  i= 5: thẻ mở 'B',                       stack=['A', 'B']
  i= 9: thẻ đóng 'A' không khớp đỉnh 'B' -> False

code = '<DIV>  unmatched <  </DIV>' (n = 26)
  i= 0: thẻ mở 'DIV',                       stack=['DIV']
  i=17: thẻ mở có tên '  </DIV' không hợp lệ -> False

code = '<A></A><B></B>' (n = 14)
  i= 0: thẻ mở 'A',                       stack=['A']
  i= 3: thẻ đóng 'A' khớp,                stack=[]
  i=7: stack rỗng nhưng còn ký tự -> False

code = '<A><![CDATA[x]]</A>' (n = 19)
  i= 0: thẻ mở 'A',                       stack=['A']
  i=3: CDATA lỗi -> False

code = '<AAAAAAAAAA></AAAAAAAAAA>' (n = 25)
  i= 0: thẻ mở có tên 'AAAAAAAAAA' không hợp lệ -> False



## 9. Code nộp LeetCode

Chọn Cách 2 (quét + stack): `O(n)`, một vòng lặp, dễ đối chiếu với từng quy tắc của đề.

In [16]:
class Solution:
    def isValid(self, code: str) -> bool:
        def ok_name(s: str) -> bool:
            return 1 <= len(s) <= 9 and all("A" <= c <= "Z" for c in s)

        st = []                                   # stack tên thẻ đang mở
        i, n = 0, len(code)
        while i < n:
            if i > 0 and not st:                  # còn ký tự ngoài thẻ gốc
                return False
            if code.startswith("<![CDATA[", i):   # CDATA: kiểm tra trước tiên
                j = code.find("]]>", i + 9)
                if not st or j == -1:
                    return False
                i = j + 3
            elif code.startswith("</", i):        # thẻ đóng
                j = code.find(">", i + 2)
                if j == -1 or not st or st[-1] != code[i + 2:j]:
                    return False
                st.pop()
                i = j + 1
            elif code[i] == "<":                  # thẻ mở
                j = code.find(">", i + 1)
                if j == -1 or not ok_name(code[i + 1:j]):
                    return False
                st.append(code[i + 1:j])
                i = j + 1
            else:                                 # ký tự thường
                if not st:
                    return False
                i += 1
        return not st

sol = Solution()
for code, exp in EXAMPLES:
    assert sol.isValid(code) == exp, code
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Bổ đề (phân tích đơn định).** Tại mỗi vị trí `i`, đề quy định duy nhất một cách hiểu: nếu bắt đầu bằng `<![CDATA[` thì là CDATA (nằm trong content), nếu bắt đầu bằng `</` thì là thẻ đóng, nếu bắt đầu bằng `<` thì là thẻ mở với tên tới `>` gần nhất (quy tắc 6), ngược lại là ký tự thường. Vậy mã hợp lệ có **đúng một** cách phân tích, và thuật toán đi theo đúng cách đó.

**Bất biến của Cách 2.** Sau khi xử lý tiền tố `code[0:i]` mà chưa trả `False`:
- tiền tố có dạng `<T1>c1<T2>c2 ... <Tk>ck` (các thẻ mở chưa đóng `T1..Tk` nằm trong stack theo thứ tự), mỗi `cj` là content hợp lệ (đã đóng đủ các thẻ con);
- nếu `k = 0` thì `i = 0` hoặc tiền tố là đúng một thẻ đóng hợp lệ hoàn chỉnh.

Mỗi nhánh duy trì bất biến: thẻ mở hợp lệ đẩy thêm `T_{k+1}`; thẻ đóng khớp đỉnh đóng `Tk` lại thành một thẻ con hợp lệ của content cha; CDATA và ký tự thường thêm vào content `ck` (chỉ khi `k >= 1`). Mọi vi phạm (tên sai, không có `>`/`]]>`, không khớp, nội dung ngoài thẻ gốc) đều tương ứng một quy tắc của đề → trả `False` là đúng.

Kết thúc: trả `True` ⇔ stack rỗng ⇔ toàn chuỗi là đúng một thẻ đóng hợp lệ. ∎

**Cách 3** là cài đặt trực tiếp của ngữ pháp nên đúng theo định nghĩa. **Cách 1**: mỗi bước xóa một thẻ trong cùng hợp lệ (content chỉ còn chữ/CDATA đã thay) — đúng là "đóng gói" một thẻ con; mã hợp lệ ⇔ có thể xóa dần tới khi còn một thẻ gốc.

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Regex rút gọn | `O(n^2)` (tối đa `O(n)` vòng `re.sub`) | `O(n)` | ngắn nhưng khó gỡ lỗi |
| 2. Stack | `O(n)` | `O(n)` | **nộp bài** |
| 3. Recursive descent | `O(n)` | `O(độ sâu)` | sát ngữ pháp nhất |

Đo trên chuỗi lồng sâu dài ~500 ký tự:

In [17]:
import time
deep = "<A>" * 70 + "x" + "</A>" * 70
print("độ dài:", len(deep))
for name, f in [("regex", is_valid_regex), ("stack", is_valid_stack), ("rd", is_valid_rd)]:
    t = time.perf_counter()
    for _ in range(200):
        r = f(deep)
    print(f"{name:>6}: {r}  {time.perf_counter() - t:.3f}s cho 200 lần")

độ dài: 491
 regex: True  0.045s cho 200 lần
 stack: True  0.013s cho 200 lần
    rd: True  0.012s cho 200 lần


## 12. Bộ test

Không có "brute force" hiển nhiên cho bài parsing, nên ta dùng **3 cài đặt độc lập** (regex, stack, đệ quy) để đối chiếu nhau trên: ví dụ đề, edge case, chuỗi ngẫu nhiên từ các "mảnh" token, và chuỗi **hợp lệ được sinh ra rồi đột biến** (để có nhiều trường hợp đúng lẫn gần-đúng).

In [18]:
import random

IMPLS = {
    "regex": is_valid_regex,
    "stack": is_valid_stack,
    "rd": is_valid_rd,
    "SolutionStack": lambda c: SolutionStack().isValid(c),
    "SolutionRD": lambda c: SolutionRD().isValid(c),
    "Solution": lambda c: Solution().isValid(c),
}

def check(code, exp=None):
    res = {name: f(code) for name, f in IMPLS.items()}
    vals = set(res.values())
    assert len(vals) == 1, (code, res)
    if exp is not None:
        assert vals == {exp}, (code, res)
    return vals.pop()

for code, exp in EXAMPLES:
    check(code, exp)

edge = {
    "<A></A>": True,                         # thẻ rỗng
    "<A></A><B></B>": False,                 # hai thẻ gốc
    " <A></A>": False,                       # ký tự trước thẻ gốc
    "<A></A> ": False,                       # ký tự sau thẻ gốc
    "<![CDATA[x]]>": False,                  # CDATA ngoài thẻ
    "<A><![CDATA[</A>]]></A>": True,         # thẻ đóng giả trong CDATA
    "<A>]]></A>": True,                      # ']]>' lẻ là chữ thường
    "<A><![CDATA[x]]</A>": False,            # CDATA không đóng
    "<ABCDEFGHI></ABCDEFGHI>": True,         # tên dài 9
    "<ABCDEFGHIJ></ABCDEFGHIJ>": False,      # tên dài 10
    "<A><></A>": False,                      # tên rỗng
    "<A>< /A>": False,                       # '< /A' là tên không hợp lệ
    "<A><B></B></A>": True,
    "<A><B></A></B>": False,
    "<A>>></A>": True,                       # '>' lẻ là chữ thường
    "<A>": False,
    "<": False,
    "t": False,
    "<A>t</A>": True,                        # chữ 't' không làm hỏng Cách 1
}
for code, exp in edge.items():
    check(code, exp)
print("Ví dụ đề + edge case OK")

Ví dụ đề + edge case OK


In [19]:
def gen_valid(rng, depth=0):
    name = "".join(rng.choice("AB") for _ in range(rng.randint(1, 3)))
    parts = []
    for _ in range(rng.randint(0, 3)):
        r = rng.random()
        if r < 0.3 and depth < 3:
            parts.append(gen_valid(rng, depth + 1))
        elif r < 0.5:
            parts.append("<![CDATA[" + "".join(rng.choice("<>/A]x[") for _ in range(rng.randint(0, 4))) + "]]>")
        else:
            parts.append("".join(rng.choice("xy >]![t") for _ in range(rng.randint(1, 3))))
    return "<" + name + ">" + "".join(parts) + "</" + name + ">"

def mutate(rng, s):
    s = list(s)
    for _ in range(rng.randint(1, 2)):
        op, p = rng.random(), rng.randrange(len(s) + 1)
        if op < 0.4 and s:
            del s[min(p, len(s) - 1)]
        elif op < 0.8:
            s.insert(p, rng.choice("<>/AB![]x"))
        else:
            s.insert(p, rng.choice(["<A>", "</A>", "<![CDATA[", "]]>"]))
    return "".join(s)

rng = random.Random(591)
tokens = ["<", ">", "/", "A", "B", "<A>", "</A>", "<B>", "</B>", "<![CDATA[", "]]>", "x", " ", "!", "[", "]", "<AB>", "</AB>", "<a>"]
cnt_true = 0
for _ in range(15000):                              # chuỗi ngẫu nhiên từ token
    s = "".join(rng.choice(tokens) for _ in range(rng.randint(1, 10)))
    cnt_true += check(s)
for _ in range(8000):                               # chuỗi hợp lệ + đột biến
    s = gen_valid(rng)
    if rng.random() < 0.6:
        s = mutate(rng, s)
    else:
        assert check(s) is True                     # chưa đột biến thì phải hợp lệ
    cnt_true += check(s)
print(f"23000 chuỗi ngẫu nhiên: 6 cài đặt khớp nhau ({cnt_true} chuỗi hợp lệ)")

23000 chuỗi ngẫu nhiên: 6 cài đặt khớp nhau (3786 chuỗi hợp lệ)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Kiểm tra thẻ mở trước CDATA

In [20]:
def wrong_order(code):
    st, i, n = [], 0, len(code)
    while i < n:
        if i > 0 and not st:
            return False
        if code.startswith("</", i):
            j = code.find(">", i + 2)
            if j == -1 or not st or st[-1] != code[i + 2:j]:
                return False
            st.pop(); i = j + 1
        elif code[i] == "<":                         # SAI: bắt cả '<![CDATA[' ở đây
            j = code.find(">", i + 1)
            if j == -1 or not valid_name(code[i + 1:j]):
                return False
            st.append(code[i + 1:j]); i = j + 1
        elif code.startswith("<![CDATA[", i):        # không bao giờ tới được
            j = code.find("]]>", i + 9)
            i = j + 3
        else:
            if not st:
                return False
            i += 1
    return not st

c = EXAMPLES[0][0]
print("sai:", wrong_order(c), "| đúng:", Solution().isValid(c))

sai: False | đúng: True


**Vì sao sai:** `<![CDATA[` bắt đầu bằng `<` nên rơi vào nhánh thẻ mở, tên `"![CDATA[<div"` không hợp lệ → `False`.

**Code đúng:** kiểm tra `<![CDATA[` **đầu tiên**, rồi `</`, rồi `<`.

### Lỗi 2 — Quên kiểm tra "còn ký tự sau khi đóng thẻ gốc"

In [21]:
def wrong_multi_root(code):
    st, i, n = [], 0, len(code)
    while i < n:
        if code.startswith("<![CDATA[", i):
            j = code.find("]]>", i + 9)
            if not st or j == -1:
                return False
            i = j + 3
        elif code.startswith("</", i):
            j = code.find(">", i + 2)
            if j == -1 or not st or st[-1] != code[i + 2:j]:
                return False
            st.pop(); i = j + 1
        elif code[i] == "<":
            j = code.find(">", i + 1)
            if j == -1 or not valid_name(code[i + 1:j]):
                return False
            st.append(code[i + 1:j]); i = j + 1
        else:
            i += 1                                  # SAI: không kiểm tra stack rỗng
    return not st

for c in ["<A></A><B></B>", "x<A></A>"]:
    print(repr(c), "sai:", wrong_multi_root(c), "| đúng:", Solution().isValid(c))

'<A></A><B></B>' sai: True | đúng: False
'x<A></A>' sai: True | đúng: False


**Vì sao sai:** quy tắc 1 yêu cầu **toàn bộ** mã nằm trong **một** thẻ. Hai thẻ cạnh nhau hoặc chữ ngoài thẻ đều phải sai.

**Code đúng:** `if i > 0 and not st: return False` ở đầu vòng lặp, và chữ thường cũng cần `st` khác rỗng.

### Lỗi 3 — Dùng `isupper()` để kiểm tra tên thẻ

In [22]:
def bad_name(s):
    return 1 <= len(s) <= 9 and s.isupper()

print("tên 'A1':", bad_name("A1"), "| đúng:", valid_name("A1"))
print("tên 'A B':", bad_name("A B"), "| đúng:", valid_name("A B"))

tên 'A1': True | đúng: False
tên 'A B': True | đúng: False


**Vì sao sai:** `isupper()` chỉ yêu cầu có ít nhất một chữ cái và mọi chữ cái đều hoa — chữ số, dấu cách vẫn lọt qua. `"<A1></A1>"` phải là sai.

**Code đúng:** `all('A' <= c <= 'Z' for c in s)`.

### Lỗi 4 — Regex CDATA tham lam (`.*` thay vì `.*?`)

In [23]:
def wrong_greedy(code):
    code = re.sub(r"<!\[CDATA\[.*\]\]>|t", "-", code)          # SAI: tham lam
    prev = None
    while code != prev:
        prev = code
        code = re.sub(r"<([A-Z]{1,9})>[^<]*</\1>", "t", code)
    return code == "t"

c = "<A><![CDATA[x]]><B></C>]]></A>"
print("sai:", wrong_greedy(c), "| đúng:", Solution().isValid(c))

sai: True | đúng: False


**Vì sao sai:** CDATA kết thúc ở `]]>` **đầu tiên**. `.*` nuốt tới `]]>` cuối cùng → `<B></C>` bị coi là nội dung CDATA và được bỏ qua.

**Code đúng:** `.*?` (lười), hoặc `find("]]>", i + 9)` như Cách 2.

### Lỗi 5 — Không kiểm tra `find` trả về `-1`

Nếu bỏ `if j == -1`, `code[i+1:-1]` vẫn chạy nhưng cắt sai chỗ, và `i = j + 1 = 0` đưa vòng lặp về đầu chuỗi → **lặp vô hạn** hoặc kết quả vô nghĩa. Luôn xử lý `-1` ngay sau `find`.

## 14. Mẹo & nhận dạng pattern

- **Đề dài nhiều quy tắc** → viết lại thành bảng "nếu thấy … thì …" (mục 4.1), rồi mỗi dòng thành một nhánh `if`.
- **Thứ tự nhánh**: tiền tố **dài/cụ thể hơn** kiểm tra trước (`<![CDATA[` trước `</` trước `<`).
- **Cặp mở/đóng có tên** → stack chứa tên.
- **Cấu trúc lồng có ngữ pháp rõ ràng** → recursive descent: mỗi luật một hàm, hàm trả về vị trí tiếp theo.
- **Không có brute force?** Viết 2–3 cài đặt **theo cách nghĩ khác nhau** và cho chúng đối chiếu nhau trên dữ liệu ngẫu nhiên + dữ liệu hợp lệ bị đột biến.

## 15. Biến thể

1. **Valid Parentheses (LeetCode 20):** phiên bản đơn giản nhất của stack mở/đóng.
2. **Cho phép nhiều thẻ gốc** (`"<A></A><B></B>"` hợp lệ): bỏ kiểm tra `i > 0 and not st`.
3. **Thẻ tự đóng** `<BR/>`: thêm nhánh nhận diện tên kết thúc bằng `/`.
4. **Trả về độ sâu lồng tối đa** của mã hợp lệ (Bài tập 5).
5. **Mini Parser (LeetCode 385), Parse Lisp Expression (LeetCode 736):** cùng kỹ thuật recursive descent.

In [24]:
# Biến thể 2: cho phép nhiều thẻ gốc liên tiếp
def is_valid_multi_root(code):
    n, i = len(code), 0
    if n == 0:
        return False
    while i < n:
        # tái sử dụng Cách 3: đọc một thẻ gốc tại i
        sub_end = None
        for end in range(i + 1, n + 1):
            if is_valid_rd(code[i:end]):
                sub_end = end
                break
        if sub_end is None:
            return False
        i = sub_end
    return True

print(is_valid_multi_root("<A></A><B>x</B>"), is_valid_multi_root("<A></A>x"))
assert is_valid_multi_root("<A></A><B>x</B>") and not is_valid_multi_root("<A></A>x")

True False


(Đoạn trên cố ý viết "ngây thơ" `O(n^2)` bằng cách tái sử dụng Cách 3; một cài đặt tốt hơn chỉ cần sửa Cách 2 như mô tả ở biến thể 2.)

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Regex rút gọn | thay CDATA, lặp xóa thẻ trong cùng, còn `"t"` là đúng | `O(n^2)` | `O(n)` | viết nhanh, kiểm chứng chéo |
| 2. Stack | quét trái → phải, bảng 4 nhánh, stack tên | `O(n)` | `O(n)` | **nộp bài** |
| 3. Recursive descent | hàm `parse_tag(i)` theo ngữ pháp | `O(n)` | `O(độ sâu)` | đề có ngữ pháp rõ, dễ mở rộng |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: kết quả của `"<A><B></B><![CDATA[<]]></A>"`, `"<A><b></b></A>"`, `"<A></A >"`?

**Bài 2 (tính tay).** Với `code = "<DIV>>>  ![cdata[]] <![CDATA[<div>]>]]>]]>>]</DIV>"`, hãy xác định vị trí bắt đầu CDATA, vị trí `]]>` đầu tiên sau nó, và `CDATA_CONTENT`.

**Bài 3 (điền TODO).** Hoàn thiện `classify(code, i)` trả về `"cdata"`, `"close"`, `"open"` hoặc `"text"` tùy theo cấu trúc bắt đầu ở `i` (đúng thứ tự ưu tiên).

**Bài 4 (tự viết lại).** Tự viết lại Cách 2 (`my_is_valid`) mà không nhìn code.

**Bài 5 (mini challenge).** Viết `max_depth(code)` trả về độ sâu lồng tối đa của thẻ nếu mã hợp lệ, ngược lại trả `-1`. (Thẻ gốc có độ sâu 1; thẻ trong CDATA không tính.)

In [25]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def classify(code, i):
    # TODO: kiểm tra '<![CDATA[' trước, rồi '</', rồi '<', còn lại là 'text'
    pass

# Bài 4 — TODO
def my_is_valid(code):
    # TODO: stack + 4 nhánh
    pass

# Bài 5 — TODO
def max_depth(code):
    # TODO: như Cách 2, theo dõi max(len(stack))
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: chữ `<` nằm trong CDATA có sao không?
- Mức 2: tên `b` có hợp lệ không?
- Mức 3: `"</A >"` — tên thẻ đóng là `"A "` (có dấu cách).

**Bài 2.**
- Mức 1: dùng `code.find("<![CDATA[")`.
- Mức 2: tìm `]]>` từ vị trí bắt đầu + 9.
- Mức 3: `CDATA_CONTENT = code[start + 9 : end]`.

**Bài 3.**
- Mức 1: dùng `startswith(…, i)`.
- Mức 2: thứ tự: CDATA → close → open → text.
- Mức 3: `code[i] == '<'` cho open.

**Bài 4.**
- Mức 1: bắt đầu từ bảng ở mục 4.1.
- Mức 2: đừng quên `i > 0 and not st`.
- Mức 3: đối chiếu với `Solution` trên `gen_valid` + `mutate`.

**Bài 5.**
- Mức 1: độ sâu = `len(st)` ngay sau khi push.
- Mức 2: nếu không hợp lệ thì trả `-1`.
- Mức 3: có thể gọi `Solution().isValid` trước, rồi quét lại chỉ để đo độ sâu.

## 19. Đáp án tham khảo

**Bài 1:** `True` (`<` trong CDATA là chữ), `False` (tên `b` không in hoa), `False` (tên thẻ đóng `"A "` không khớp `"A"`).

**Bài 2:** CDATA bắt đầu ở vị trí 20; `]]>` đầu tiên ở vị trí 36; `CDATA_CONTENT = "<div>]>"`.

In [26]:
assert Solution().isValid("<A><B></B><![CDATA[<]]></A>") is True
assert Solution().isValid("<A><b></b></A>") is False
assert Solution().isValid("<A></A >") is False

c = "<DIV>>>  ![cdata[]] <![CDATA[<div>]>]]>]]>>]</DIV>"
start = c.find("<![CDATA[")
end = c.find("]]>", start + 9)
print(start, end, repr(c[start + 9:end]))
assert (start, end, c[start + 9:end]) == (20, 36, "<div>]>")

# Bài 3
def classify(code, i):
    if code.startswith("<![CDATA[", i):
        return "cdata"
    if code.startswith("</", i):
        return "close"
    if code[i] == "<":
        return "open"
    return "text"

assert [classify("<A></A><![CDATA[", i) for i in (0, 3, 7, 1)] == ["open", "close", "cdata", "text"]
print("Bài 1, 2, 3 OK")

20 36 '<div>]>'
Bài 1, 2, 3 OK


In [27]:
# Bài 4
def my_is_valid(code):
    st, i = [], 0
    while i < len(code):
        if i and not st:
            return False
        kind = classify(code, i)
        if kind == "cdata":
            j = code.find("]]>", i + 9)
            if j < 0 or not st:
                return False
            i = j + 3
        elif kind == "close":
            j = code.find(">", i + 2)
            if j < 0 or not st or st.pop() != code[i + 2:j]:
                return False
            i = j + 1
        elif kind == "open":
            j = code.find(">", i + 1)
            if j < 0 or not valid_name(code[i + 1:j]):
                return False
            st.append(code[i + 1:j])
            i = j + 1
        else:
            if not st:
                return False
            i += 1
    return not st

# Bài 5
def max_depth(code):
    if not Solution().isValid(code):
        return -1
    st, i, best = [], 0, 0
    while i < len(code):
        kind = classify(code, i)
        if kind == "cdata":
            i = code.find("]]>", i + 9) + 3
        elif kind == "close":
            st.pop()
            i = code.find(">", i + 2) + 1
        elif kind == "open":
            j = code.find(">", i + 1)
            st.append(code[i + 1:j])
            best = max(best, len(st))
            i = j + 1
        else:
            i += 1
    return best

rng = random.Random(5)
for _ in range(3000):
    s = gen_valid(rng)
    if rng.random() < 0.5:
        s = mutate(rng, s)
    assert my_is_valid(s) == Solution().isValid(s), s
assert max_depth("<A><B><C></C></B><![CDATA[<D><E>]]></A>") == 3
assert max_depth("<A></A>") == 1 and max_depth("<a></a>") == -1
print("Bài 4, 5 OK")

Bài 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "quét + stack cho thẻ":**

```python
st, i, n = [], 0, len(code)
while i < n:
    if i > 0 and not st: return False            # ngoài thẻ gốc
    if code.startswith("<![CDATA[", i):           # 1. CDATA
        j = code.find("]]>", i + 9)
        if not st or j == -1: return False
        i = j + 3
    elif code.startswith("</", i):                # 2. thẻ đóng
        j = code.find(">", i + 2)
        if j == -1 or not st or st[-1] != code[i+2:j]: return False
        st.pop(); i = j + 1
    elif code[i] == "<":                          # 3. thẻ mở
        j = code.find(">", i + 1)
        if j == -1 or not ok_name(code[i+1:j]): return False
        st.append(code[i+1:j]); i = j + 1
    else:                                         # 4. chữ thường
        if not st: return False
        i += 1
return not st
```

**Câu thần chú:** *"CDATA trước, đóng rồi mở; tên in hoa 1–9; `find` nhớ check -1; gốc chỉ một."*

**Checklist trước khi submit:**

- [ ] Thứ tự nhánh: `<![CDATA[` → `</` → `<` → chữ.
- [ ] Mọi `find` đều kiểm tra `-1`.
- [ ] Tên thẻ: 1..9 ký tự, **mỗi** ký tự trong `'A'..'Z'`.
- [ ] CDATA kết thúc ở `]]>` **đầu tiên**; CDATA ngoài thẻ là sai.
- [ ] Không có gì trước/sau thẻ gốc; không có hai thẻ gốc.
- [ ] Hết chuỗi thì stack rỗng.
- [ ] Đã thử mọi ví dụ trong đề (cả ví dụ sai).